# RAG：从可控召回到权限与指标反例

本实验使用可控 embedding，不下载模型。你将先预测 BM25/dense 的排序，再观察 RRF，随后故意混入另一个租户的结果，并用错误检索结果验证 Recall/MRR 会怎样变化。

## 1. 建立一个能解释的最小语料

三篇 course 文档分别讲 KV Cache、RAG 和 LoRA；第四篇与 RAG 语义相同，却属于 other tenant。可控向量让排序原因完全可见。

运行前预测：查询“外部证据”时，dense 检索最容易混淆哪两篇文档？ACL 应在哪一层阻止它？

In [ ]:
from collections.abc import Sequence

import numpy as np

from about_llm.rag import BM25Index, DenseIndex, Document, reciprocal_rank_fusion

documents = [
    Document('kv', 'KV Cache 保存历史 token 的 key 和 value', 'course'),
    Document('rag', 'RAG 先检索证据再生成回答', 'course'),
    Document('lora', 'LoRA 学习低秩权重增量', 'course'),
    Document('secret', '另一个租户的机密 RAG 文档', 'other'),
]

class ControlledEmbedder:
    def __init__(self, values): self.values = values
    def encode(self, texts: Sequence[str]):
        return np.asarray([self.values[text] for text in texts], dtype=np.float32)

vectors = {
    documents[0].text: (1.0, 0.0, 0.0),
    documents[1].text: (0.0, 1.0, 0.0),
    documents[2].text: (0.0, 0.0, 1.0),
    documents[3].text: (0.0, 1.0, 0.0),
    '如何避免重复计算注意力？': (0.9, 0.1, 0.0),
    '怎样用外部证据回答？': (0.1, 0.9, 0.0),
}
bm25 = BM25Index(documents)
dense = DenseIndex(documents, ControlledEmbedder(vectors))


## 2. 分开观察 lexical、semantic 与 fusion

不要只看最终 RRF 第一名。先比较两路各自找到什么，再确认进入融合的候选已经属于目标 tenant。

In [ ]:
query = '怎样用外部证据回答？'
lexical = bm25.search(query, tenant_id='course', top_k=3)
semantic = dense.search(query, tenant_id='course', top_k=3)
fused = reciprocal_rank_fusion([lexical, semantic], top_k=3)

def show(name, results):
    print(name, [(item.document.document_id, round(item.score, 4)) for item in results])

show('BM25', lexical)
show('Dense', semantic)
show('RRF', fused)
assert all(item.document.tenant_id == 'course' for item in fused)


## 3. 故意跨租户融合，观察为什么“最后过滤”太晚

下面模拟应用层误把两个 tenant 的已检索列表交给同一个 RRF。程序会正常返回结果，但 secret 文档已经参与排序。

In [ ]:
other_semantic = dense.search(query, tenant_id='other', top_k=3)
unsafe_fused = reciprocal_rank_fusion([semantic, other_semantic], top_k=4)
unsafe_ids = [item.document.document_id for item in unsafe_fused]
print('unsafe cross-tenant fusion:', unsafe_ids)
assert 'secret' in unsafe_ids

safe_ids = [item.document.document_id for item in fused]
print('safe tenant-scoped fusion:', safe_ids)
assert 'secret' not in safe_ids

## 4. 先写 relevant set，再计算排名指标

Recall@k 回答相关文档是否进入前 k；MRR@k 还关心第一个相关文档的位置。两者都依赖 query ID、相关性标签和明确分母。

In [ ]:
from about_llm.evaluation import mean_reciprocal_rank, recall_at_k

queries = {
    'q-kv': '如何避免重复计算注意力？',
    'q-rag': '怎样用外部证据回答？',
}
relevant = {'q-kv': {'kv'}, 'q-rag': {'rag'}}
retrieved = {}
for query_id, text in queries.items():
    dense_results = dense.search(text, tenant_id='course', top_k=2)
    retrieved[query_id] = [item.document.document_id for item in dense_results]

print('retrieved:', retrieved)
print('Recall@2:', recall_at_k(retrieved, relevant, k=2))
print('MRR@2:', mean_reciprocal_rank(retrieved, relevant, k=2))
assert recall_at_k(retrieved, relevant, k=2) == 1.0
assert mean_reciprocal_rank(retrieved, relevant, k=2) == 1.0


## 5. 制造一个检索错误，确认指标会失败

如果指标在明显错误的结果上仍是 1，通常是 case 对齐、relevant set 或分母出了问题。

In [ ]:
broken_retrieved = dict(retrieved)
broken_retrieved['q-rag'] = ['kv']
broken_recall = recall_at_k(broken_retrieved, relevant, k=2)
broken_mrr = mean_reciprocal_rank(broken_retrieved, relevant, k=2)
print('broken Recall@2:', broken_recall)
print('broken MRR@2:', broken_mrr)
assert broken_recall < 1.0
assert broken_mrr < 1.0

## 6. 把召回结论限制在正确边界

已经证明：固定小语料上的 BM25/dense/RRF 排序可观察；tenant-scoped retrieval 不返回 other 文档；错误地跨租户融合会立刻污染候选；指标能对一个故意错误的 case 下降。

尚未证明：真实 embedding 质量、reranker、上下文 packing、引用忠实度、无答案拒答、生产 ACL 或延迟。

继续实验：替换一个变量并保留对照。例如固定语料只换 embedding，或固定召回只换 chunk size。最终答案错误时，分别保存 retrieval、rerank、packed context 和 generation evidence。